# 係り受けのパターンで特定の構文を探す (DependencyMatcher)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_parsing/dependency_matcher.ipynb)

NLP クックブック 13-06 ／ 構文解析

連体修飾節、受身、使役、並列のような特定の構文を含む箇所を、係り受けの形を指定してコーパスから取り出したい。

In [1]:
!pip install -q ginza ja_ginza
!python -m spacy download en_core_web_sm -q

In [2]:
import spacy
from spacy.matcher import DependencyMatcher

nlp = spacy.load("ja_ginza")
matcher = DependencyMatcher(nlp.vocab)

# 各パターンは「基準のトークン」と、そこからの関係 (> は直接の子、< は直接の親) で書く
matcher.add("連体修飾節", [[
    {"RIGHT_ID": "noun", "RIGHT_ATTRS": {"POS": {"IN": ["NOUN", "PROPN"]}}},
    {"LEFT_ID": "noun", "REL_OP": ">", "RIGHT_ID": "pred", "RIGHT_ATTRS": {"DEP": "acl"}},
]])
matcher.add("受身", [[
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"POS": "VERB"}},
    {"LEFT_ID": "verb", "REL_OP": ">", "RIGHT_ID": "aux", "RIGHT_ATTRS": {"LEMMA": {"IN": ["れる", "られる"]}}},
]])
matcher.add("使役", [[
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"POS": "VERB"}},
    {"LEFT_ID": "verb", "REL_OP": ">", "RIGHT_ID": "aux", "RIGHT_ATTRS": {"LEMMA": {"IN": ["せる", "させる"]}}},
]])
matcher.add("名詞の並列", [[
    {"RIGHT_ID": "second", "RIGHT_ATTRS": {"POS": {"IN": ["NOUN", "PROPN"]}}},
    {"LEFT_ID": "second", "REL_OP": ">", "RIGHT_ID": "first", "RIGHT_ATTRS": {"DEP": "nmod"}},
    {"LEFT_ID": "first", "REL_OP": ">", "RIGHT_ID": "particle", "RIGHT_ATTRS": {"ORTH": {"IN": ["と", "や", "および", "または"]}}},
]])

texts = ["太郎が昨日図書館で借りた本を花子が読んでいる。", "太郎は先生に褒められた。", "母は子供に野菜を食べさせた。",
         "りんごとみかんを買った。", "弟に大切なケーキを食べられた。", "走っている犬や寝ている猫を見た。"]
for doc in nlp.pipe(texts):
    print(doc.text)
    for match_id, ids in matcher(doc):
        print("   ", nlp.vocab.strings[match_id], [doc[i].text for i in ids])

太郎が昨日図書館で借りた本を花子が読んでいる。
    連体修飾節 ['本', '借り']
太郎は先生に褒められた。
    受身 ['褒め', 'られ']
母は子供に野菜を食べさせた。
    使役 ['食べ', 'させ']
りんごとみかんを買った。
    名詞の並列 ['みかん', 'りんご', 'と']
弟に大切なケーキを食べられた。
    連体修飾節 ['ケーキ', '大切']
    受身 ['食べ', 'られ']
走っている犬や寝ている猫を見た。
    連体修飾節 ['犬', '走っ']
    連体修飾節 ['猫', '寝']
    名詞の並列 ['猫', '犬', 'や']


In [3]:
# 英語: 受動態と関係節
nlp_en = spacy.load("en_core_web_sm")
m = DependencyMatcher(nlp_en.vocab)
m.add("passive", [[
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"POS": "VERB"}},
    {"LEFT_ID": "verb", "REL_OP": ">", "RIGHT_ID": "subj", "RIGHT_ATTRS": {"DEP": "nsubjpass"}},
]])
m.add("relative clause", [[
    {"RIGHT_ID": "noun", "RIGHT_ATTRS": {"POS": {"IN": ["NOUN", "PROPN"]}}},
    {"LEFT_ID": "noun", "REL_OP": ">", "RIGHT_ID": "verb", "RIGHT_ATTRS": {"DEP": "relcl"}},
]])
doc = nlp_en("The paper that she wrote was rejected by the reviewers. The model was trained on data which we collected.")
for match_id, ids in m(doc):
    print(nlp_en.vocab.strings[match_id], [doc[i].text for i in ids])

passive ['rejected', 'paper']
passive ['trained', 'model']
relative clause ['paper', 'wrote']
relative clause ['data', 'collected']


- `REL_OP` には `>` (直接の子)、`<` (直接の親)、`>>` (子孫)、`<<` (祖先)、`.` (直後)、`;` (直前) などがある。
- 日本語の UD では、名詞の並列は `conj` ではなく「と」「や」の付いた `nmod` として表される。パターンを書く前に、狙った構文が実際にどう解析されるかを確認する。
- 「られる」は受身のほかに可能・尊敬・自発の用法があり、係り受けの形だけでは区別できない。
- 表層の語順で書くなら `Matcher`、係り受けの形で書くなら `DependencyMatcher`。語順が自由な日本語では後者が向いている場面が多い。